In [1]:
import os
import ast
import copy
import difflib
import tempfile
import subprocess
import pandas as pd
from tqdm import tqdm
import shutil

In [2]:
# === CONFIGURATION ===
code_dir = 'RQ3_1_FAISS_Fewshot_Prompt2_codesnippets'
test_dir = 'RQ3_1_FAISS_Fewshot_Prompt2_testscripts'
results_csv = 'mutation_results_FAISS.csv'
timeout_secs = 120
parent_mutant_dir = 'mutation_mutants_FAISS'
os.makedirs(parent_mutant_dir, exist_ok=True)

In [3]:
# === Mutation Operator Mappings ===
comparison_map = {
    ast.Gt: ast.LtE,
    ast.LtE: ast.Gt,
    ast.Lt: ast.GtE,
    ast.GtE: ast.Lt,
    ast.Eq: ast.NotEq,
    ast.NotEq: ast.Eq
}

boolean_map = {
    ast.And: ast.Or,
    ast.Or: ast.And
}

arithmetic_map = {
    ast.Add: ast.Sub,
    ast.Sub: ast.Add,
    ast.Mult: ast.Div,
    ast.Div: ast.Mult
}

def get_code_lines(code):
    return code.splitlines(keepends=True)

def diff_code(original, mutant):
    original_lines = get_code_lines(original)
    mutant_lines = get_code_lines(mutant)
    diff = difflib.unified_diff(original_lines, mutant_lines, fromfile='original', tofile='mutant', lineterm='')
    return '\n'.join([line for line in diff if not line.startswith(('---', '+++', '@@'))])

def generate_mutants(tree, original_code, code_filename):
    mutants = []

    class Mutator(ast.NodeTransformer):
        def visit_Compare(self, node):
            for i, op in enumerate(node.ops):
                op_type = type(op)
                if op_type in comparison_map:
                    mutant_tree = copy.deepcopy(tree)
                    class ReplaceCompare(ast.NodeTransformer):
                        def visit_Compare(self, n):
                            if n.lineno == node.lineno and n.col_offset == node.col_offset:
                                n.ops[i] = comparison_map[op_type]()
                            return n
                    mutant_tree = ReplaceCompare().visit(mutant_tree)
                    ast.fix_missing_locations(mutant_tree)
                    desc = f"Flip Compare {op_type.__name__} ➝ {comparison_map[op_type].__name__} at line {node.lineno}"
                    mutants.append((mutant_tree, desc, node.lineno))
            return node

        def visit_BoolOp(self, node):
            op_type = type(node.op)
            if op_type in boolean_map:
                mutant_tree = copy.deepcopy(tree)
                class ReplaceBoolOp(ast.NodeTransformer):
                    def visit_BoolOp(self, n):
                        if n.lineno == node.lineno and n.col_offset == node.col_offset:
                            n.op = boolean_map[op_type]()
                        return n
                mutant_tree = ReplaceBoolOp().visit(mutant_tree)
                ast.fix_missing_locations(mutant_tree)
                desc = f"Flip BoolOp {op_type.__name__} ➝ {boolean_map[op_type].__name__} at line {node.lineno}"
                mutants.append((mutant_tree, desc, node.lineno))
            return node

        def visit_BinOp(self, node):
            op_type = type(node.op)
            if op_type in arithmetic_map:
                mutant_tree = copy.deepcopy(tree)
                class ReplaceBinOp(ast.NodeTransformer):
                    def visit_BinOp(self, n):
                        if n.lineno == node.lineno and n.col_offset == node.col_offset:
                            n.op = arithmetic_map[op_type]()
                        return n
                mutant_tree = ReplaceBinOp().visit(mutant_tree)
                ast.fix_missing_locations(mutant_tree)
                desc = f"Flip BinOp {op_type.__name__} ➝ {arithmetic_map[op_type].__name__} at line {node.lineno}"
                mutants.append((mutant_tree, desc, node.lineno))
            return node

    Mutator().visit(tree)
    print(f"[LOG] {len(mutants)} mutation points in {code_filename}")
    return mutants

def run_tests(test_file, mutant_file_path):
    # Copy mutant code file into the test folder under its original name
    target_file = os.path.join(os.path.dirname(test_file), os.path.basename(mutant_file_path))

    # Use shutil instead of 'cp' for cross-platform compatibility
    shutil.copyfile(mutant_file_path, target_file)

    try:
        result = subprocess.run(
            ['pytest', test_file, '--maxfail=1', '--disable-warnings', '--tb=short'],
            capture_output=True,
            text=True,
            timeout=timeout_secs  # 2-minute timeout
        )
    except subprocess.TimeoutExpired:
        print(f"⏱️ Timeout: {test_file} with mutant took more than {timeout_secs} seconds.")
        os.remove(target_file)
        return False

    os.remove(target_file)
    return result.returncode != 0  # True if mutant is killed

In [4]:
TIMEOUT = 120

# === Main Execution Loop ===
results = []
code_files = sorted([f for f in os.listdir(code_dir) if f.startswith('code_snippet_') and f.endswith('.py')])
print(f"📂 Found {len(code_files)} code files.")

for i in tqdm(range(len(code_files)), desc="Mutation Testing"):
    code_file = os.path.join(code_dir, f'code_snippet_{i}.py')
    test_file = os.path.join(test_dir, f'test_code_{i}.py')

    if not os.path.exists(code_file) or not os.path.exists(test_file):
        results.append({
            'Code File': f'code_snippet_{i}.py', 
            'Test Script': f'test_code_{i}.py',
            'Total Mutants': 0,
            'Killed Mutants': 0,
            'Mutation Score (%)': 'Missing File'
        })
        continue

    with open(code_file, 'r', encoding='utf-8') as f:
        original_code = f.read()

    try:
        tree = ast.parse(original_code)
    except SyntaxError:
        results.append({
            'Code File': f'code_snippet_{i}.py',
            'Test Script': f'test_code_{i}.py',
            'Total Mutants': 0,
            'Killed Mutants': 0,
            'Mutation Score (%)': 'SyntaxError'
        })
        continue

    mutants = generate_mutants(tree, original_code, f'code_snippet_{i}.py')
    total = len(mutants)
    killed = 0

    mutant_diff_dir = os.path.join(parent_mutant_dir, f'mutants_diff_code_{i}')
    os.makedirs(mutant_diff_dir, exist_ok=True)

    for idx, (mutant_tree, desc, line) in enumerate(mutants):
        mutant_code = ast.unparse(mutant_tree)
        diff_text = diff_code(original_code, mutant_code)
    
        diff_file = os.path.join(mutant_diff_dir, f'mutant_{idx+1}_line{line}.txt')
        with open(diff_file, 'w', encoding='utf-8') as df:
            df.write(f"# Mutation: {desc}\n\n{diff_text}")
    
        with tempfile.NamedTemporaryFile('w', delete=False, suffix='.py') as tmp_mut_file:
            tmp_mut_file.write(mutant_code)
            tmp_path = tmp_mut_file.name
    
        timed_out = False
        try:
            if run_tests(test_file, tmp_path):
                killed += 1
        except subprocess.TimeoutExpired:
            print(f"⏱️ Timeout: {test_file} with mutant {idx+1} took more than {timeout_secs} seconds. Skipping remaining mutants.")
            timed_out = True
        finally:
            os.remove(tmp_path)
    
        if timed_out:
            break  # Skip to next script

    score = round((killed / total) * 100, 2) if total > 0 else 0.0
    results.append({
        'Code File': f'code_snippet_{i}.py',
        'Test Script': f'test_code_{i}.py',
        'Total Mutants': total,
        'Killed Mutants': killed,
        'Mutation Score (%)': score
    })

📂 Found 598 code files.


Mutation Testing:   0%|                                                                                     | 0/598 [00:00<?, ?it/s]

[LOG] 1 mutation points in code_snippet_0.py


Mutation Testing:   0%|▏                                                                            | 1/598 [00:00<07:40,  1.30it/s]

[LOG] 1 mutation points in code_snippet_1.py


...

Mutation Testing: 100%|████████████████████████████████████████████████████████████████████████▊| 596/598 [1:08:16<00:15,  7.54s/it]

[LOG] 2 mutation points in code_snippet_596.py


Mutation Testing: 100%|████████████████████████████████████████████████████████████████████████▉| 597/598 [1:08:19<00:06,  6.10s/it]

[LOG] 5 mutation points in code_snippet_597.py


Mutation Testing: 100%|█████████████████████████████████████████████████████████████████████████| 598/598 [1:08:23<00:00,  6.86s/it]


In [5]:
# === Save Results ===
df = pd.DataFrame(results)
df.to_csv(results_csv, index=False)
print(f"\n✅ Mutation testing complete. Results saved to: {results_csv}")

# === Compute Average Mutation Score ===
df['Mutation Score (%)'] = pd.to_numeric(df['Mutation Score (%)'], errors='coerce')
valid_scores = df['Mutation Score (%)'].dropna()
average_score = valid_scores.mean() if not valid_scores.empty else 0.0
print(f"📊 Average Mutation Score: {average_score:.2f}%")


✅ Mutation testing complete. Results saved to: mutation_results_FAISS.csv
📊 Average Mutation Score: 50.86%
